# Manga / Webtoon → 대사 추출 + 번역 (Colab)

파이프라인: `이미지/PDF → 대사 위치 찾기 → OCR → 언어 확인 → 외국어만 한국어 번역 → JSONL/TXT`

- 단일 이미지 / 여러 이미지 / PDF를 자동으로 구분합니다.
- 업로드 직후 썸네일로 입력을 확인할 수 있습니다.
- 한국어는 번역하지 않고 그대로 출력합니다.
- 일본어는 기본적으로 MangaOCR, 한국어/중국어/영어는 PaddleOCR을 사용합니다.
- PaddleOCR은 **CPU**, RF-DETR과 Qwen 번역 모델은 **Colab GPU(T4)** 를 사용합니다.
- 각 단계마다 진단 로그를 출력해서 문제가 생긴 위치를 찾기 쉽게 했습니다.

> 이전 실행에서 `paddlepaddle-gpu`를 설치했다면, Colab 메뉴에서 **런타임 → 세션 다시 시작** 후 이 노트북을 처음부터 실행하는 것을 권장합니다.


## 1. 패키지 설치

Colab 기본 PyTorch/CUDA 환경은 그대로 둡니다.

PaddleOCR은 GPU 버전을 설치하지 않고 **CPU용 PaddlePaddle 3.2.2**를 사용합니다.  
이렇게 하면 PyTorch CUDA와 Paddle CUDA 라이브러리가 충돌하지 않습니다.


In [ ]:
!pip uninstall -y -q \
    paddlepaddle \
    paddlepaddle-gpu \
    paddleocr \
    paddlex

!pip install -q \
    "paddlepaddle==3.2.2" \
    "paddleocr==3.3.2" \
    "paddlex==3.3.13" \
    "rfdetr==1.7.0" \
    "safetensors>=0.5" \
    "huggingface_hub>=0.27" \
    "manga-ocr>=0.1.11" \
    "transformers>=4.51" \
    "accelerate>=1.2" \
    "bitsandbytes>=0.45" \
    "lingua-language-detector>=2.0" \
    "pymupdf>=1.24" \
    pillow \
    numpy \
    tqdm \
    matplotlib

!rm -rf /content/manga2text_tmp
!git clone -q \
    https://github.com/HisameOgasahara/manga2text_tmp.git \
    /content/manga2text_tmp

print("[설치 완료]")


## 2. 실행 환경 확인

문제가 생기면 이 셀의 출력부터 확인하면 됩니다.

정상적인 Colab T4 환경이라면:
- `PyTorch CUDA 사용 가능: True`
- GPU 이름에 `T4`
- `Paddle CUDA 빌드: False`
처럼 나오는 것이 정상입니다.


In [ ]:
import platform

import paddle
import paddleocr
import paddlex
import torch

print("[실행 환경]")
print("Python 버전           :", platform.python_version())
print("PyTorch 버전          :", torch.__version__)
print("PyTorch CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름              :", torch.cuda.get_device_name(0))

print("Paddle 버전           :", paddle.__version__)
print("PaddleOCR 버전        :", paddleocr.__version__)
print("PaddleX 버전          :", paddlex.__version__)
print("Paddle CUDA 빌드      :", paddle.device.is_compiled_with_cuda())
print("Paddle 현재 장치      :", paddle.device.get_device())

if paddle.device.is_compiled_with_cuda():
    print("[주의] 이 노트북은 PaddleOCR을 CPU로 쓰도록 설계했습니다.")


## 3. 사용자 설정

아래 항목만 바꾸면 됩니다.

- **언어 자동 판별**: 켜면 앞부분 대사를 샘플로 읽어서 한국어/일본어/중국어/영어를 자동 판단합니다.
- **원문 언어**: 자동 판별을 끈 경우에만 사용합니다.
- **글자 읽는 방법**: `자동`이면 일본어는 MangaOCR, 나머지는 PaddleOCR을 사용합니다.
- **읽기 방향**: `자동`이면 일본어 만화는 오른쪽→왼쪽, 나머지는 왼쪽→오른쪽으로 처리합니다.


In [ ]:
import sys
from pathlib import Path

sys.path.append("/content/manga2text_tmp")

from manga2text_pipeline import (
    auto_detect_source_language,
    build_language_detector,
    classify_inputs,
    collect_page_images,
    describe_input_mode,
    load_koharu_detector,
    load_ocr_backend,
    load_translation_model,
    make_preview_images,
    process_pages,
    resolve_ocr_configuration,
    save_results,
)

# @title 사용자 설정

언어_자동_판별 = True  # @param {type:"boolean"}
원문_언어 = "한국어"  # @param ["한국어", "일본어", "중국어", "영어"]

글자_읽는_방법 = "자동"  # @param ["자동", "MangaOCR", "PaddleOCR"]
읽기_방향 = "자동"  # @param ["자동", "오른쪽→왼쪽 (일본 만화)", "왼쪽→오른쪽 (웹툰/영문)"]

외국어_한국어_번역 = True  # @param {type:"boolean"}
번역_모델 = "Qwen3-1.7B (가볍고 빠름)"  # @param ["Qwen3-1.7B (가볍고 빠름)", "Qwen3-4B (품질 우선)"]

효과음도_읽기 = False  # @param {type:"boolean"}

PDF_화질_DPI = 200  # @param {type:"integer"}
처리할_페이지_수 = 0  # @param {type:"integer"}
동시에_준비할_작업_수 = 4  # @param {type:"integer"}

진단_로그_보기 = True  # @param {type:"boolean"}

# ------------------------------------------------------------------
# 위의 쉬운 한국어 설정을 실제 코드용 값으로 바꿉니다.
# ------------------------------------------------------------------

언어_코드 = {
    "한국어": "ko",
    "일본어": "ja",
    "중국어": "zh",
    "영어": "en",
}

OCR_코드 = {
    "자동": "auto",
    "MangaOCR": "manga",
    "PaddleOCR": "paddle",
}

읽기_방향_코드 = {
    "자동": "auto",
    "오른쪽→왼쪽 (일본 만화)": "rtl",
    "왼쪽→오른쪽 (웹툰/영문)": "ltr",
}

번역_모델_코드 = {
    "Qwen3-1.7B (가볍고 빠름)": "Qwen/Qwen3-1.7B",
    "Qwen3-4B (품질 우선)": "Qwen/Qwen3-4B",
}

AUTO_DETECT_SOURCE_LANGUAGE = 언어_자동_판별
SOURCE_LANGUAGE = 언어_코드[원문_언어]

OCR_BACKEND = OCR_코드[글자_읽는_방법]
READING_DIRECTION = 읽기_방향_코드[읽기_방향]

ENABLE_TRANSLATION = 외국어_한국어_번역
TRANSLATION_MODEL = 번역_모델_코드[번역_모델]

INCLUDE_SFX = 효과음도_읽기

PDF_DPI = PDF_화질_DPI
PAGE_LIMIT = None if 처리할_페이지_수 <= 0 else 처리할_페이지_수
INPUT_WORKERS = max(1, 동시에_준비할_작업_수)

DEBUG_LOG = 진단_로그_보기

# PaddleOCR은 CPU 고정입니다.
PADDLE_DEVICE = "cpu"

# 보통 바꿀 필요 없는 내부 설정입니다.
MAX_NEW_TOKENS = 256
CROP_PADDING = 8
ROW_TOLERANCE = 80
AUTO_LANGUAGE_SAMPLE_CROPS = 3
DEBUG_SAMPLES_PER_PAGE = 3

CLASS_THRESHOLDS = {
    0: 0.25,  # 일반 텍스트
    1: 0.20,  # 효과음
    2: 0.50,  # 말풍선
    3: 0.50,  # 만화 칸
}

WORK_DIR = Path("/content/manga2text")
INPUT_DIR = WORK_DIR / "input"
PAGE_DIR = WORK_DIR / "pages"
OUTPUT_DIR = WORK_DIR / "output"

for directory in [INPUT_DIR, PAGE_DIR, OUTPUT_DIR]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("[현재 설정]")
print("언어 자동 판별       :", AUTO_DETECT_SOURCE_LANGUAGE)
print("수동 원문 언어       :", 원문_언어)
print("글자 읽는 방법       :", 글자_읽는_방법)
print("읽기 방향            :", 읽기_방향)
print("PaddleOCR 계산 장치  : CPU")
print("외국어 번역          :", ENABLE_TRANSLATION)
print("번역 모델            :", 번역_모델)
print("효과음 읽기          :", INCLUDE_SFX)
print("PDF 화질             :", PDF_DPI, "DPI")
print("페이지 제한          :", PAGE_LIMIT if PAGE_LIMIT is not None else "전체")
print("동시 준비 작업 수    :", INPUT_WORKERS)
print("진단 로그            :", DEBUG_LOG)


## 4. 만화 업로드 + 자동 판별 + 썸네일 확인

한 번에 다음 입력을 모두 지원합니다.

- 이미지 1장
- 이미지 여러 장
- PDF 1개 또는 여러 개
- 이미지 + PDF 혼합

이미지 여러 장은 **파일명 순서**로 처리하므로 `001.jpg`, `002.jpg`, `003.jpg`처럼 이름을 맞추는 것이 좋습니다.


In [ ]:
import shutil

import matplotlib.pyplot as plt
from google.colab import files

# 이전 업로드가 남아 있지 않도록 입력 폴더만 비웁니다.
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

uploaded_files = files.upload()

for filename, file_bytes in uploaded_files.items():
    destination = INPUT_DIR / filename
    destination.write_bytes(file_bytes)

input_groups = classify_inputs(INPUT_DIR)
input_mode = describe_input_mode(input_groups)

print("[입력 확인]")
print("입력 종류        :", input_mode)
print("이미지 수        :", len(input_groups["images"]))
print("PDF 수           :", len(input_groups["pdfs"]))
print("지원하지 않는 수 :", len(input_groups["unsupported"]))

for path in input_groups["unsupported"]:
    print("[경고] 지원하지 않는 파일:", path.name)

preview_items = make_preview_images(
    input_dir=INPUT_DIR,
    max_items=8,
    pdf_preview_pages=3,
)

if not preview_items:
    raise RuntimeError(
        "처리할 수 있는 이미지 또는 PDF가 없습니다."
    )

column_count = min(
    4,
    len(preview_items),
)
row_count = (
    len(preview_items) + column_count - 1
) // column_count

plt.figure(
    figsize=(4 * column_count, 5 * row_count)
)

for index, (label, image) in enumerate(
    preview_items,
    start=1,
):
    plt.subplot(
        row_count,
        column_count,
        index,
    )
    plt.imshow(image)
    plt.title(label)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 5. 페이지 이미지 준비

PDF는 페이지별 이미지로 변환합니다.

여러 이미지 복사와 PDF 페이지 변환은 CPU에서 여러 작업으로 나누어 처리합니다.  
GPU 모델 추론은 뒤 단계에서 순차적으로 실행합니다.


In [ ]:
if PAGE_DIR.exists():
    shutil.rmtree(PAGE_DIR)

PAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("[페이지 준비]")
print("동시 작업 수 :", INPUT_WORKERS)
print("PDF 화질    :", PDF_DPI, "DPI")
print(
    "페이지 제한 :",
    PAGE_LIMIT if PAGE_LIMIT is not None else "전체",
)

page_paths = collect_page_images(
    input_dir=INPUT_DIR,
    page_dir=PAGE_DIR,
    pdf_dpi=PDF_DPI,
    page_limit=PAGE_LIMIT,
    workers=INPUT_WORKERS,
)

print("준비된 페이지 수:", len(page_paths))

for path in page_paths[:10]:
    print(" -", path)

if not page_paths:
    raise RuntimeError(
        "처리할 페이지가 없습니다."
    )


## 6. 대사 위치 찾기 모델(RF-DETR) 로드

이 모델은 글자를 읽는 모델이 아닙니다.

만화 페이지에서 **텍스트 / 효과음 / 말풍선 / 만화 칸의 위치**를 찾습니다.  
Colab GPU(T4)를 사용합니다.


In [ ]:
print("[대사 위치 모델 로드]")

detector = load_koharu_detector()

print("[대사 위치 모델 상태]")
print("모델 :", "Koharu Layout RF-DETR Seg 2XL")
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print(
        "[경고] GPU가 잡히지 않았습니다. "
        "Colab 런타임 유형을 GPU로 설정하세요."
    )


## 7. 원문 언어와 OCR 방식 결정

### 자동 판별을 켠 경우
앞부분의 텍스트 영역 몇 개를 가져와서:
- 한국어 PaddleOCR
- 일본어 MangaOCR
- 중국어 PaddleOCR
- 영어 PaddleOCR

을 시험합니다.

각 후보가 읽은 문자열과 점수를 출력하므로 자동 판별이 틀렸을 때 확인하기 쉽습니다.

### 자동 판별을 끈 경우
3번 셀에서 고른 `원문 언어`를 그대로 사용합니다.


In [ ]:
if AUTO_DETECT_SOURCE_LANGUAGE:
    selected_source_language, auto_language_details = (
        auto_detect_source_language(
            page_paths=page_paths,
            detector=detector,
            class_thresholds=CLASS_THRESHOLDS,
            crop_padding=CROP_PADDING,
            max_crops=AUTO_LANGUAGE_SAMPLE_CROPS,
            paddle_device=PADDLE_DEVICE,
        )
    )
else:
    selected_source_language = SOURCE_LANGUAGE
    auto_language_details = None

    print("[언어 수동 선택]")
    print("선택한 언어:", 원문_언어)

ocr_config = resolve_ocr_configuration(
    source_language=selected_source_language,
    ocr_backend=OCR_BACKEND,
    reading_direction=READING_DIRECTION,
)

print("[결정된 처리 방법]")
print(
    "원문 언어      :",
    ocr_config["source_language"],
)
print(
    "사용할 OCR     :",
    ocr_config["ocr_backend"],
)
print(
    "PaddleOCR 언어 :",
    ocr_config["paddle_lang"],
)
print(
    "읽기 방향      :",
    ocr_config["reading_direction"],
)
print(
    "PaddleOCR 장치  :",
    PADDLE_DEVICE,
)


## 8. OCR 모델 로드

- 일본어 + 자동 설정 → MangaOCR
- 한국어/중국어/영어 + 자동 설정 → PaddleOCR
- PaddleOCR은 CPU에서 실행합니다.


In [ ]:
print("[OCR 모델 로드]")

ocr_model = load_ocr_backend(
    backend=ocr_config["ocr_backend"],
    paddle_lang=ocr_config["paddle_lang"],
    paddle_device=PADDLE_DEVICE,
)

print("[OCR 모델 로드 완료]")
print("OCR 방식:", ocr_config["ocr_backend"])

if ocr_config["ocr_backend"] == "paddle":
    print("계산 장치:", "CPU")


## 9. 언어 확인기 + 번역 모델 로드

OCR로 읽은 문자열이 실제로 한국어인지 다시 확인합니다.

- 한국어 → 그대로 출력
- 일본어 / 중국어 / 영어 → Qwen으로 한국어 번역

번역 모델은 4비트로 불러와 Colab GPU 메모리 사용량을 줄입니다.


In [ ]:
language_detector, language_to_code = (
    build_language_detector()
)

translation_tokenizer = None
translation_model = None

if ENABLE_TRANSLATION:
    print("[번역 모델 로드]")

    translation_tokenizer, translation_model = (
        load_translation_model(
            model_name=TRANSLATION_MODEL,
        )
    )

    print("[번역 모델 로드 완료]")
    print("모델:", TRANSLATION_MODEL)
else:
    print("[번역 비활성화]")


## 10. 전체 파이프라인 실행

각 페이지에서 다음 순서로 처리합니다.

1. RF-DETR이 텍스트 위치를 찾음
2. OCR이 실제 글자를 읽음
3. 언어를 확인함
4. 한국어가 아니면 번역함
5. 결과를 기록함

`진단_로그_보기=True`이면 각 페이지의 검출 개수, OCR 샘플, 언어, 번역 여부가 출력됩니다.


In [ ]:
records = process_pages(
    page_paths=page_paths,
    detector=detector,
    ocr_backend=ocr_config["ocr_backend"],
    ocr_model=ocr_model,
    language_detector=language_detector,
    language_to_code=language_to_code,
    class_thresholds=CLASS_THRESHOLDS,
    reading_direction=ocr_config["reading_direction"],
    row_tolerance=ROW_TOLERANCE,
    crop_padding=CROP_PADDING,
    include_sfx=INCLUDE_SFX,
    enable_translation=ENABLE_TRANSLATION,
    translation_tokenizer=translation_tokenizer,
    translation_model=translation_model,
    max_new_tokens=MAX_NEW_TOKENS,
    debug=DEBUG_LOG,
    debug_samples_per_page=DEBUG_SAMPLES_PER_PAGE,
)

print("[처리 완료]")
print("추출된 대사 수:", len(records))


## 11. 결과 미리보기

원문과 최종 한국어 결과를 나란히 확인합니다.


In [ ]:
preview_count = min(
    30,
    len(records),
)

for record in records[:preview_count]:
    page_number = record["page"]
    order = record["order"]
    language = record["language"]
    original_text = record["original"]
    korean_text = record["korean"]

    print(
        f"[페이지 {page_number:03d} / 순서 {order:02d}] "
        f"언어={language}"
    )
    print("원문 :", original_text)
    print("결과 :", korean_text)
    print()


## 12. JSONL / TXT 저장 + 다운로드

- `dialogues.jsonl`: 페이지 번호, 위치, OCR 원문, 언어, 번역 결과까지 모두 저장
- `dialogues.txt`: 최종 한국어 대사만 페이지별로 저장


In [ ]:
from google.colab import files

jsonl_path, txt_path = save_results(
    records=records,
    output_dir=OUTPUT_DIR,
)

print("[저장 완료]")
print("JSONL:", jsonl_path)
print("TXT  :", txt_path)

files.download(
    str(jsonl_path)
)

files.download(
    str(txt_path)
)
